# Milestone M0: Data Reconnaissance
## Regional Landslide Risk & Emergency Response Decision System (SIH26001)
**Course**: CSE326 / CSE276  
**Blueprint**: Version 2.0  
**Objective**: Systematic reconnaissance of public candidate datasets for Region A (e.g., Western Ghats / Kerala / Nilgiris) and Region B (e.g., Himalayas / Uttarakhand / Himachal Pradesh) without performing unverified large downloads.

> **SAFETY RESTRICTIONS**:
> - Do not auto-download large raster/grid archives.
> - All download helpers run in `dry_run=True` mode by default.
> - Verify endpoints with lightweight HTTP `HEAD` probes or metadata queries only.
> - Document access credentials, API keys, licensing, and expected volumes before proceeding to acquisition.

### 1. Environment Setup & Module Imports

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
cwd = Path(".").resolve()
PROJECT_ROOT = cwd if (cwd / "configs").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
import pandas as pd
import numpy as np

from src.data.ingestion import (
    DataSource,
    describe_source,
    probe_url,
    safe_download,
    list_candidate_sources,
    DownloadAbortedError,
)
from src.data.validation import (
    run_all_checks,
    check_01_schema,
    check_02_crs,
    check_03_null_fraction,
    check_04_value_ranges,
    check_05_duplicates,
    check_06_spatial_bounds,
    check_08_date_precision,
    check_09_class_imbalance,
    check_10_temporal_coverage,
    check_11_dem_voids,
    check_12_sm_tier_consistency,
)

print(f"Project root resolved to: {PROJECT_ROOT}")
print("Modules imported successfully.")

### 2. Inspect Configuration & Provisional Candidates

In [ ]:
config_path = PROJECT_ROOT / "configs" / "config.yaml"
with open(config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("=== PROJECT CONFIGURATION SUMMARY ===")
print(f"Project: {cfg.get('project', {}).get('name')} ({cfg.get('project', {}).get('id')})")
print(f"Regions configured: {list(cfg.get('regions', {}).keys())}")

for r_code, r_info in cfg.get("regions", {}).items():
    print(f"\nRegion [{r_code}]: {r_info.get('name')}")
    print(f"  Target Districts: {r_info.get('candidate_districts', [])}")
    print(f"  BBox [min_lon, min_lat, max_lon, max_lat]: {r_info.get('bounding_box')}")
    print(f"  Projected CRS: {r_info.get('utm_zone')}")

### 3. Registry of Candidate Data Sources
Review candidate public repositories across four domains:
1. **Landslide Catalogs**: NASA Global Landslide Catalog (GLC/COOLR), GSI open portal, state disaster archives.
2. **Digital Elevation Models (DEM)**: Copernicus GLO-30 (ESA open), SRTM 30m, ALOS AW3D30.
3. **Precipitation**: CHIRPS 2.0 (0.05° daily, open HTTP), GPM IMERG Final (0.1°, EarthData login), ERA5-Land (ECMWF CDS).
4. **Soil Moisture / Texture**: NASA SMAP L3 (Tier 1), GLEAM v3 / GLDAS (Tier 2), HWSD v2 / SoilGrids (Tier 3 fallback).

In [ ]:
candidate_sources = list_candidate_sources()
df_sources = pd.DataFrame(candidate_sources)
display_cols = ["key", "name", "domain", "access_type", "license", "expected_size_mb", "m0_status"]
print(f"Total registered candidate sources: {len(df_sources)}")
print(df_sources[["key", "name", "access"]].to_string())

### 4. Domain 1 Reconnaissance: Landslide Event Catalogs
We investigate NASA GLC and regional catalog availability. We probe public endpoints and inspect schema requirements.

In [ ]:
describe_source(DataSource.NASA_GLC)

# Probe the NASA Open Data / GLC landing portal
glc_url = "https://data.nasa.gov/api/views/dd9e-wu2v/rows.csv?accessType=DOWNLOAD"
glc_probe = probe_url(glc_url, timeout=10)
print(f"NASA GLC Probe Result: {glc_probe}")

### 5. Domain 2 Reconnaissance: Terrain & Elevation (DEM)
Compare Copernicus GLO-30 vs SRTM 30m. Copernicus GLO-30 is preferred for 30m resolution without SRTM void artifacts in high mountain ranges.

In [ ]:
describe_source(DataSource.COPERNICUS_DEM)

# Probe public Copernicus DEM / OpenTopography portal or AWS open data bucket
copernicus_portal = "https://copernicus-dem-30m.s3.amazonaws.com/readme.html"
copernicus_probe = probe_url(copernicus_portal, timeout=10)
print(f"Copernicus GLO-30 S3 Open Data Probe: {copernicus_probe}")

### 6. Domain 3 Reconnaissance: Precipitation & Rainfall Intensity
CHIRPS provides 0.05° (~5.5 km) daily precipitation from 1981 to near real-time without authentication.

In [ ]:
describe_source(DataSource.CHIRPS_PRECIP)

# Probe CHIRPS UCSB HTTP repository root
chirps_url = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_daily/netcdf/p05/"
chirps_probe = probe_url(chirps_url, timeout=10)
print(f"CHIRPS Data Server Probe: {chirps_probe}")

### 7. Domain 4 Reconnaissance: Soil Moisture vs Soil Texture (The 3-Tier Hierarchy)
Per Blueprint v2.0 Section 8:
- **Tier 1 (Direct Observed)**: NASA SMAP L3 Enhanced (9 km, 2015-present). Requires NASA EarthData Login (free).
- **Tier 2 (Modelled Physical)**: GLEAM v3.8a / GLDAS Noah (0.25°, 1980-present). Open data.
- **Tier 3 (Static Proxy)**: HWSD v2 clay % + Topographic Wetness Index (TWI). Strictly labeled as static proxy fallback.

In [ ]:
print("--- TIER 1: NASA SMAP ---")
describe_source(DataSource.NASA_SMAP_L3)

print("\n--- TIER 3 FALLBACK: FAO/HWSD Clay Content ---")
describe_source(DataSource.HWSD_SOIL)

### 8. Safe Ingestion Verification (Dry-Run Demonstration)
Demonstrating that `safe_download` enforces:
1. `dry_run=True` refusal to write to disk without user permission
2. File size checking against pre-declared quota
3. SHA-256 integrity verification hooks

In [ ]:
# Demonstrating safe dry-run download: refuses write without explicit consent
try:
    safe_download(
        url="https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_daily/netcdf/p05/README-CHIRPS.txt",
        dest_path=PROJECT_ROOT / "data" / "raw" / "region_a" / "chirps_readme.txt",
        expected_size_mb=0.01,
        expected_sha256=None,
        dry_run=True,
    )
except DownloadAbortedError as e:
    print(f"Safe Download Guard Activated (Expected): {e}")

### 9. Validation Scaffold Verification: Category A vs Category B Checks
We verify that the 12 data-quality checks behave according to specification:
- **Category A checks** execute immediately on in-memory structural data.
- **Category B checks** (requiring downloaded rasters/shapefiles) return `SKIP` status with clear explanation instead of failing.

In [ ]:
# Construct a synthetic test DataFrame matching the planned 21-feature schema
np.random.seed(42)
dates = pd.date_range("2020-01-01", periods=50, freq="W")
synthetic_events = pd.DataFrame({
    "district_id": ["Idukki"] * 25 + ["Wayanad"] * 25,
    "date": dates,
    "region_flag": [0] * 50,  # Region A
    "target": [1] * 10 + [0] * 40,  # 20% positive class
    "slope_deg": np.random.uniform(5, 55, 50),
    "aspect_deg": np.random.uniform(0, 360, 50),
    "curvature": np.random.uniform(-1, 1, 50),
    "twi": np.random.uniform(4, 12, 50),
    "elevation_m": np.random.uniform(200, 2000, 50),
    "rain_3d_mm": np.random.uniform(10, 200, 50),
    "rain_7d_mm": np.random.uniform(20, 400, 50),
    "rain_15d_mm": np.random.uniform(50, 800, 50),
    "rain_intensity_max": np.random.uniform(5, 80, 50),
    "sm_3d_mean": np.random.uniform(0.1, 0.5, 50),
    "sm_event_day": np.random.uniform(0.1, 0.6, 50),
    "sm_tier": [1] * 50,
    "soil_clay_pct": np.random.uniform(10, 60, 50),
    "soil_texture_class": [2] * 50,
    "lulc_class": [1] * 50,
    "fault_dist_km": np.random.uniform(1, 20, 50),
    "lithology_class": [1] * 50,
    "centroid_lon": np.linspace(74.0, 77.0, 50),
    "centroid_lat": np.linspace(9.5, 12.0, 50),
    "catalog_source": ["NASA_GLC"] * 10 + [None] * 40,
    "date_precision": ["day"] * 10 + [None] * 40,
})

print(f"Synthetic test frame shape: {synthetic_events.shape}")

# Execute full validation suite in reconnaissance mode
recon_report = run_all_checks(synthetic_events, cfg, region_code="region_a")
recon_report.print_summary()

### 10. Summary & Output Checklist for Milestone M0
Results from this notebook are transferred into `reports/data_reconnaissance_report.md`.
No models have been trained. No full datasets have been downloaded.
All decisions remain subject to review.